<a href="https://colab.research.google.com/github/shahaanabbas-13/Quantum-Diabetes-Prediction/blob/main/notebooks/04_Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Feature selection and dimensionality reduction
Compare Mutual Information, RFE and PCA. All selectors are fitted only on training folds in a pipeline. This notebook reports CV comparisons for a logistic-regression reference model.

In [ ]:
%pip -q install scikit-learn

In [1]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_validate


# Separate features and target
X = df.drop(columns=TARGET).copy()
y = df[TARGET].astype(int).copy()

# Treat invalid zeros as missing
X[INVALID_ZERO_COLS] = X[INVALID_ZERO_COLS].replace(0, np.nan)

# Same 80:20 stratified split as Notebook 03
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

# Five-fold training-only cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

selector_options = {
    "All 8 features": "passthrough",
    "Mutual Information (4)": SelectKBest(
        score_func=mutual_info_classif,
        k=4
    ),
    "RFE (4)": RFE(
        estimator=LogisticRegression(max_iter=2000),
        n_features_to_select=4
    ),
    "PCA (4)": PCA(
        n_components=4,
        random_state=42
    )
}

rows = []

for name, selector in selector_options.items():

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("selector", selector),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "recall": "recall",
            "f1": "f1",
            "roc_auc": "roc_auc"
        },
        n_jobs=-1
    )

    rows.append({
        "representation": name,
        **{
            f"{metric}_mean": scores[f"test_{metric}"].mean()
            for metric in ["accuracy", "recall", "f1", "roc_auc"]
        },
        **{
            f"{metric}_std": scores[f"test_{metric}"].std()
            for metric in ["accuracy", "recall", "f1", "roc_auc"]
        }
    })

feature_results = pd.DataFrame(rows)

display(feature_results)

# Save results
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "feature_selection_cv.csv"

feature_results.to_csv(RESULTS_PATH, index=False)

print("Feature selection results saved successfully!")
print("Location:", RESULTS_PATH)

display(pd.read_csv(RESULTS_PATH))

Training data: (614, 8)
Testing data: (154, 8)


,representation,accuracy_mean,recall_mean,f1_mean,roc_auc_mean,accuracy_std,recall_std,f1_std,roc_auc_std
0,All 8 features,0.762202,0.700997,0.672385,0.844381,0.011053,0.039478,0.013319,0.016074
1,Mutual Information (4),0.755684,0.710188,0.669672,0.833915,0.012886,0.019865,0.012305,0.019703
2,RFE (4),0.771985,0.729125,0.690115,0.842306,0.013625,0.042225,0.014537,0.018942
3,PCA (4),0.732907,0.687154,0.640040,0.826476,0.024237,0.080559,0.046569,0.009926


Feature selection results saved successfully!
Location: /content/results/feature_selection_cv.csv


,representation,accuracy_mean,recall_mean,f1_mean,roc_auc_mean,accuracy_std,recall_std,f1_std,roc_auc_std
0,All 8 features,0.762202,0.700997,0.672385,0.844381,0.011053,0.039478,0.013319,0.016074
1,Mutual Information (4),0.755684,0.710188,0.669672,0.833915,0.012886,0.019865,0.012305,0.019703
2,RFE (4),0.771985,0.729125,0.690115,0.842306,0.013625,0.042225,0.014537,0.018942
3,PCA (4),0.732907,0.687154,0.640040,0.826476,0.024237,0.080559,0.046569,0.009926


For quantum circuits, use a four-dimensional representation. Mutual Information and RFE retain original features; PCA creates transformed components. Fit selectors within each training fold to avoid leakage.

In [5]:

import base64
import requests
from google.colab import userdata

# GitHub configuration
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
OWNER = "shahaanabbas-13"
REPO = "Quantum-Diabetes-Prediction"
BRANCH = "main"

FILE_PATH = "results/classical_cv.csv"
LOCAL_FILE = "/content/results/feature_selection_cv.csv"

API_URL = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{FILE_PATH}"

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# Read the local CSV
with open(LOCAL_FILE, "rb") as f:
    content = f.read()

encoded_content = base64.b64encode(content).decode("utf-8")

# Check whether the file already exists on GitHub
check = requests.get(
    API_URL,
    headers=headers,
    params={"ref": BRANCH}
)

payload = {
    "message": "Update classical ML cross-validation results",
    "content": encoded_content,
    "branch": BRANCH
}

if check.status_code == 200:
    # Existing file: include its SHA
    payload["sha"] = check.json()["sha"]
    print("Existing file found. Updating it...")

elif check.status_code == 404:
    # New file: no SHA needed
    print("File not found. Creating a new file...")

else:
    raise Exception(
        f"GitHub file check failed: {check.status_code}\n{check.text}"
    )

# Upload
response = requests.put(
    API_URL,
    headers=headers,
    json=payload
)

print("Upload status:", response.status_code)

if response.status_code in [200, 201]:
    print("Classical CV results uploaded successfully!")
    print("GitHub path:", FILE_PATH)
else:
    print(response.text)


Existing file found. Updating it...
Upload status: 200
Classical CV results uploaded successfully!
GitHub path: results/classical_cv.csv
